# Week 4 — Customer Segmentation with RFM

**Course:** Data Science for Business (YSU) · Customer Behavior Analysis  
**Prerequisite:** cleaned transactions from Lecture 2 (`data/data_cleared.csv`)  
**Next:** [Week 4.1 — Clustering](./Week_4.1_Segmentation_with_Clustering.ipynb)

---

### Learning objectives

1. Explain **why** we segment customers and what RFM measures.
2. Collapse line-item data to **customer × order** grain.
3. Compute **Recency, Frequency, Monetary** and score customers with quartiles.
4. Read RFM distributions and segment charts to recommend actions.

### Agenda (~45 min)

| Block | Topic |
|---|---|
| 1 | Segmentation motivation + RFM framework |
| 2 | Build the customer-level RFM table |
| 3 | Score customers (quartiles + RFM string) |
| 4 | Visualize segments and interpret results |


## 1. Why segment customers?

**Customer segmentation** divides customers into groups with similar behavior so marketing, product, and support can treat each group differently.

**Why it matters**

- Find your best and worst customers.
- Tailor campaigns (loyalty vs win-back).
- Spot product gaps and service issues.
- Improve upsell / cross-sell targeting.

> RFM is a classic, rule-based method. Later notebooks use **clustering** on the same data for comparison.


## 2. What is RFM?

**RFM** scores each customer on three dimensions:

| Letter | Question | Better when… |
|---|---|---|
| **R — Recency** | How long since the last purchase? | …they bought **recently** (fewer days) |
| **F — Frequency** | How many orders did they place? | …they order **often** |
| **M — Monetary** | How much did they spend in total? | …they spend **more** |

The Pareto idea: a small share of customers often drives a large share of revenue. RFM makes that visible.

*Reference: [CleverTap — RFM Analysis](https://clevertap.com/blog/rfm-analysis/)*


## 3. Load data

We use the same cleaned Online Retail file from [Lecture 2](../Lecture%202/Week_1.2_Data_Preparation_and_EDA.ipynb).

Each row is still a **line item**. RFM needs **one row per customer**, so our first transform is:

> **customer × order date** → sum spend per visit, then aggregate to customer level.


In [ ]:
from pathlib import Path
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline


def _acq_weights(n):
    w = np.linspace(1.4, 0.7, n)
    w[0] *= 1.6
    return w / w.sum()


def make_synthetic_retail(n_customers=1800, seed=5) -> pd.DataFrame:
    """Fallback data when data/data_cleared.csv is missing."""
    rng = np.random.default_rng(seed)
    months = pd.period_range("2010-12", "2011-12", freq="M")
    first = rng.choice(months[:-1], size=n_customers, p=_acq_weights(len(months) - 1))
    rows = []
    invoice = 10000
    for i, start in enumerate(first):
        cid = 10000 + i
        quality = rng.uniform(0.15, 0.55)
        first_spend = float(rng.lognormal(3.4, 0.7))
        for m in months[months >= start]:
            age = (m - start).n
            if age == 0:
                p_buy = 1.0
            else:
                p_buy = quality * (0.72 ** max(age - 1, 0))
                p_buy *= 1.25 if m.month == 12 else 1.0
                p_buy = min(p_buy, 0.95)
            if rng.random() > p_buy:
                continue
            n_lines = int(rng.integers(1, 5))
            spend = first_spend if age == 0 else first_spend * rng.uniform(0.4, 1.3)
            for _ in range(n_lines):
                invoice += 1
                qty = int(rng.integers(1, 8))
                rows.append(
                    {
                        "InvoiceNo": invoice,
                        "InvoiceDate": m.to_timestamp()
                        + pd.Timedelta(days=int(rng.integers(0, 27))),
                        "CustomerID": cid,
                        "Quantity": qty,
                        "TotalPrice": spend / n_lines,
                    }
                )
    return pd.DataFrame(rows)


def load_transactions() -> pd.DataFrame:
    for path in [Path("data/data_cleared.csv"), Path("../data/data_cleared.csv")]:
        if path.exists():
            df = pd.read_csv(path)
            print(f"Loaded {len(df):,} line items from {path.resolve()}")
            return df
    print("data/data_cleared.csv not found — using synthetic retail data.")
    return make_synthetic_retail()


data = load_transactions()
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
data["CustomerID"] = pd.to_numeric(data["CustomerID"], errors="coerce")
data = data.dropna(subset=["CustomerID", "InvoiceDate"]).copy()
data["CustomerID"] = data["CustomerID"].astype("int64")
print(
    f"{data['CustomerID'].nunique():,} customers | "
    f"{data['InvoiceDate'].min().date()} to {data['InvoiceDate'].max().date()}"
)
data.head()

In [ ]:
# RFM concept diagram (replaces missing static image)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.axis("off")
labels = [
    ("R\nRecency\n(days since last purchase)", "#4C72B0"),
    ("F\nFrequency\n(number of orders)", "#55A868"),
    ("M\nMonetary\n(total spend)", "#C44E52"),
]
for i, (text, color) in enumerate(labels):
    x = i * 3.1
    ax.add_patch(plt.Rectangle((x, 0.15), 2.7, 2.1, fc=color, alpha=0.2, ec=color, lw=2))
    ax.text(x + 1.35, 1.2, text, ha="center", va="center", fontsize=10)
ax.set_xlim(-0.1, 9.4)
ax.set_ylim(0, 2.6)
ax.set_title("RFM = three scores combined into one customer profile", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

### Step 1 — Customer × order date

One customer can buy multiple products on the same day. We treat that as **one order** and sum `TotalPrice` for the day.


In [ ]:
dt = (
    data.groupby(["CustomerID", "InvoiceDate"], as_index=False)["TotalPrice"]
    .sum()
    .rename(columns={"TotalPrice": "Budget"})
)
dt["InvoiceDate"] = pd.to_datetime(dt["InvoiceDate"])
print("Last date in dataset:", dt["InvoiceDate"].max().date())
dt.head()

### Step 2 — Compute R, F, M

We pick a fixed **snapshot date** (last date in the dataset) and measure:

- **Recency** — days between snapshot and last order
- **Frequency** — number of distinct order dates
- **Monetary** — total spend across all orders


In [ ]:
snapshot = dt["InvoiceDate"].max()

rfm = (
    dt.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
        frequency=("InvoiceDate", "count"),
        monetary=("Budget", "sum"),
    )
    .reset_index()
)

rfm.describe().round(1)

## 4. Explore the three metrics

Skewed distributions are normal in retail: many one-time buyers, a few VIPs.  
We plot histogram + KDE and mark mean (dashed) and median (solid red).


In [ ]:
def plot_rfm_distribution(series, title, xlabel):
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(series, kde=True, ax=ax, color="#4C72B0", bins=40)
    ax.axvline(series.mean(), color="black", linestyle="--", linewidth=1, label=f"mean = {series.mean():.0f}")
    ax.axvline(series.median(), color="crimson", linewidth=1.5, label=f"median = {series.median():.0f}")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_rfm_distribution(rfm["recency"], "Distribution of Recency", "Days since last purchase")
plot_rfm_distribution(rfm["frequency"], "Distribution of Frequency", "Number of orders")
plot_rfm_distribution(rfm["monetary"], "Distribution of Monetary value", "Total spend")

**Reading the plots**

- **Recency** — median ~50 days: typical customer returns within ~7 weeks; the long tail is likely churn.
- **Frequency** — most customers order only 1–2 times; heavy repeat buyers are rare.
- **Monetary** — spend is right-skewed; a few customers dominate revenue.


## 5. Quartile scoring

We assign each metric a quartile score (1 = best, 4 = worst for recency/monetary).

**Frequency special case:** up to half of customers have frequency = 1 or 2, so equal quartiles collapse. We use custom bins:

- bottom 50% → score 3 (lowest frequency)
- 50–75% → score 2
- top 25% → score 1 (highest frequency)


In [ ]:
rfm["r_quartile"] = pd.qcut(rfm["recency"], 4, labels=["1", "2", "3", "4"])
rfm["f_quartile"] = pd.qcut(rfm["frequency"], [0, 0.5, 0.75, 1.0], labels=["3", "2", "1"])
rfm["m_quartile"] = pd.qcut(rfm["monetary"], 4, labels=["4", "3", "2", "1"])

rfm["RFM_Score"] = rfm["r_quartile"].astype(str) + rfm["f_quartile"].astype(str) + rfm["m_quartile"].astype(str)
rfm.head(10)

### Named segments (business-friendly labels)

The 3-digit score is precise but hard to read. This helper maps common patterns to action labels.


In [ ]:
def rfm_segment_name(row) -> str:
    r, f, m = int(row["r_quartile"]), int(row["f_quartile"]), int(row["m_quartile"])
    if r == 1 and f == 1 and m == 1:
        return "Champions"
    if r == 1 and f <= 2:
        return "New / Promising"
    if r <= 2 and f <= 2 and m <= 2:
        return "Loyal"
    if r >= 3 and f >= 2 and m <= 2:
        return "At Risk"
    if r >= 3 and f >= 3:
        return "Lost / Churned"
    if r >= 3 and f <= 2 and m <= 2:
        return "Cannot Lose Them"
    return "Needs Attention"


rfm["Segment"] = rfm.apply(rfm_segment_name, axis=1)
rfm[["CustomerID", "recency", "frequency", "monetary", "RFM_Score", "Segment"]].head(12)

## 6. Visualize segments

Four views that marketing teams actually use:

1. **Top RFM scores** — which 3-digit codes dominate?
2. **Segment mix** — pie of named groups
3. **R × F heatmap** — where customers sit in recency/frequency space
4. **Scatter** — recency vs frequency, colored by spend quartile


In [ ]:
TOP_N = 12
seg_counts = (
    rfm.groupby("RFM_Score", as_index=False)["CustomerID"]
    .count()
    .rename(columns={"CustomerID": "Customers"})
    .sort_values("Customers", ascending=False)
    .head(TOP_N)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=seg_counts, x="RFM_Score", y="Customers", ax=ax, color="#4C72B0")
ax.set_title(f"Top {TOP_N} RFM scores by customer count")
ax.set_xlabel("RFM score (R + F + M quartiles)")
ax.set_ylabel("Number of customers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
named = rfm["Segment"].value_counts().reset_index()
named.columns = ["Segment", "Customers"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    named["Customers"],
    labels=named["Segment"],
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
)
ax.set_title("Customer mix by RFM segment")
plt.tight_layout()
plt.show()
named

In [ ]:
rf_heatmap = (
    rfm.groupby(["r_quartile", "f_quartile"], observed=True)["CustomerID"]
    .count()
    .unstack(fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(rf_heatmap, annot=True, fmt=",d", cmap="Blues", ax=ax)
ax.set_title("Customer count: Recency quartile (rows) × Frequency quartile (cols)")
ax.set_xlabel("Frequency quartile (1 = most frequent)")
ax.set_ylabel("Recency quartile (1 = most recent)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
palette = {"1": "#2ca02c", "2": "#ff7f0e", "3": "#d62728", "4": "#9467bd"}
sns.scatterplot(
    data=rfm.sample(min(2000, len(rfm)), random_state=42),
    x="recency",
    y="frequency",
    hue="m_quartile",
    palette=palette,
    alpha=0.55,
    ax=ax,
)
ax.set_title("Recency vs Frequency (sample), colored by Monetary quartile")
ax.set_xlabel("Recency (days)")
ax.set_ylabel("Frequency (orders)")
ax.legend(title="Monetary quartile", loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = named["Segment"].tolist()
sns.boxplot(data=rfm, x="Segment", y="monetary", order=order, ax=ax, color="#4C72B0")
ax.set_title("Spend distribution by named segment")
ax.set_xlabel("")
ax.set_ylabel("Total spend")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 7. Interpretation

Typical pattern in this dataset:

| Pattern | Example score | What it means | Suggested action |
|---|---|---|---|
| Best customers | **111** | Recent, frequent, high spend | Loyalty rewards, early access |
| Likely churn | **434** | Long ago, one purchase, low spend | Win-back campaign or ignore |
| High value but quiet | **311** / **411** | Good spend history, not recent | Personal outreach before they leave |

**Food for thought**

- Would you use the same quartile cutoffs for a subscription business?
- How does RFM differ from the **cohort retention** view in Lecture 3?
- In [Week 4.1](./Week_4.1_Segmentation_with_Clustering.ipynb), we cluster on RFM — do the groups match these labels?

### Practice

1. Filter `Segment == "Champions"`. What is their median monetary value?
2. Create a bar chart of **average monetary** by `Segment`.
3. Export `rfm[["CustomerID", "RFM_Score", "Segment"]]` to CSV for a CRM upload mock-up.
